# Практическая работа № 3
## Сборка пайплайна предобработки

**Цель работы:** собрать пайплайн предобработки данных с использованием `scikit-learn` и исключить утечку информации из тестовой выборки в обучающую.

**Задачи работы:**
1. Разделить данные на обучающую и тестовую выборки.
2. Реализовать собственный трансформер для обработки выбросов.
3. Обработать пропуски и категориальные признаки.
4. Объединить преобразования в `Pipeline` и `ColumnTransformer`.
5. Проверить трансформер тестами и оценить производительность на увеличенной выборке.

## Описание данных

Для работы используется учебный набор данных с информацией о возрасте, зарплате и городе проживания людей.

Данные содержат числовые и категориальные признаки, пропущенные значения и выбросы. Это позволяет проверить работу пайплайна предобработки.

Целевой признак — `purchased`, который показывает, совершил ли человек покупку.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
df = pd.DataFrame({
    "age": [22, 25, 31, 45, 120, 28, np.nan, 36, 41, 29,
            33, 27, 39, 52, 24, 30, 48, 35, 26, 44],
    "salary": [30000, 35000, 42000, 55000, 999999, 39000,
               41000, np.nan, 62000, 38000, 47000, 36000,
               51000, 70000, 32000, 43000, 65000, 49000,
               34000, 58000],
    "city": ["Bishkek", "Osh", "Bishkek", "Osh", "Bishkek",
             "Naryn", "Osh", "Bishkek", np.nan, "Naryn",
             "Osh", "Bishkek", "Naryn", "Osh", "Bishkek",
             "Osh", "Naryn", "Bishkek", "Osh", "Naryn"],
    "purchased": [0, 0, 1, 1, 1, 0, 0, 1, 1, 0,
                  1, 0, 1, 1, 0, 1, 1, 0, 0, 1]
})

df.head()

,age,salary,city,purchased
0,22.0,30000.0,Bishkek,0
1,25.0,35000.0,Osh,0
2,31.0,42000.0,Bishkek,1
3,45.0,55000.0,Osh,1
4,120.0,999999.0,Bishkek,1


In [3]:
X = df.drop("purchased", axis=1)
y = df["purchased"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Размер train:", X_train.shape)
print("Размер test:", X_test.shape) 

Размер train: (16, 3)
Размер test: (4, 3)


In [5]:
class OutlierRemover(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.lower_bounds = {}
        self.upper_bounds = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()

        for col in X.columns:
            q1 = X[col].quantile(0.25)
            q3 = X[col].quantile(0.75)
            iqr = q3 - q1

            self.lower_bounds[col] = q1 - 1.5 * iqr
            self.upper_bounds[col] = q3 + 1.5 * iqr

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        for col in X.columns:
            X[col] = X[col].clip(
                self.lower_bounds[col],
                self.upper_bounds[col]
            )

        return X

In [6]:
numeric_data = X_train[["age", "salary"]]

outlier_transformer = OutlierRemover()
outlier_transformer.fit(numeric_data)

cleaned_data = outlier_transformer.transform(numeric_data)

print("До обработки:")
display(numeric_data)

print("После обработки:")
display(cleaned_data)

До обработки:


,age,salary
2,31.0,42000.0
7,36.0,NaN
3,45.0,55000.0
0,22.0,30000.0
13,52.0,70000.0
9,29.0,38000.0
11,27.0,36000.0
19,44.0,58000.0
10,33.0,47000.0
5,28.0,39000.0


После обработки:


,age,salary
2,31.0,42000.0
7,36.0,NaN
3,45.0,55000.0
0,22.0,30000.0
13,52.0,70000.0
9,29.0,38000.0
11,27.0,36000.0
19,44.0,58000.0
10,33.0,47000.0
5,28.0,39000.0


In [7]:
numeric_features = ["age", "salary"]
categorical_features = ["city"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("outliers", OutlierRemover()),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Пайплайн предобработки создан!")

Пайплайн предобработки создан!


In [8]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Train после обработки:", X_train_processed.shape)
print("Test после обработки:", X_test_processed.shape)
print("Пропусков после обработки:", np.isnan(X_train_processed).sum())

Train после обработки: (16, 5)
Test после обработки: (4, 5)
Пропусков после обработки: 0


In [9]:
print("Размер исходного датасета:", df.shape)
print("Размер X_train:", X_train.shape)
print("Размер X_test:", X_test.shape)

print("\nПропуски после обработки:")
print("Train:", np.isnan(X_train_processed).sum())
print("Test:", np.isnan(X_test_processed).sum())

print("\nПайплайн успешно обработал данные!")

Размер исходного датасета: (20, 4)
Размер X_train: (16, 3)
Размер X_test: (4, 3)

Пропуски после обработки:
Train: 0
Test: 0

Пайплайн успешно обработал данные!
